In [1]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

/data/ll2531/venv/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
with open("processed_data_pkl/imputed_training_data.pkl", "rb") as f:
    traffic_dic = pickle.load(f)

with open("processed_data_pkl/weather_global_2014_2025.pkl", "rb") as f:
    weather = pickle.load(f)

air_qual = pd.read_csv("online_data/air_qual/air_qual.csv")

In [3]:
air_qual.drop(columns=["aerosol_optical_depth ()", "dust (μg/m³)"], inplace=True)
air_qual.dropna(inplace=True)
air_qual.reset_index(inplace=True, drop=True)

In [4]:
air_qual["time"] = pd.to_datetime(air_qual["time"], errors="coerce")
weather["timestamp"] = pd.to_datetime(weather["timestamp"], errors="coerce")

weather = weather[weather["timestamp"].isin(air_qual["time"])]
weather.reset_index(inplace=True, drop=True)

In [5]:
dataset = pd.concat([weather, air_qual], axis=1)
dataset.drop(columns=["time"], inplace=True)

# setting up the graph

In [6]:
import networkx as nx
import torch
from torch_geometric.utils import to_networkx
from torch_geometric.nn import GCNConv
from torch.nn import Linear
import geopy.distance
from torch_geometric.data import Data


/data/ll2531/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
G = nx.DiGraph()

for sensor_id, directions in traffic_dic.items():
    for direction, df in directions.items():
        origin_lat = df["latitude"].iloc[0]
        origin_lon = df["longitude"].iloc[0]

        G.add_node(sensor_id, pos=(origin_lon, origin_lat))
        
        for sensor_id_2, directions_2 in traffic_dic.items():
            if sensor_id == sensor_id_2:
                continue

            for direction_2, df_2 in directions_2.items():
                target_lat = df_2["latitude"].iloc[0]
                target_lon = df_2["longitude"].iloc[0]
                
                distance_m = geopy.distance.geodesic(
                    (origin_lat, origin_lon), 
                    (target_lat, target_lon)
                ).m

                if distance_m < 1000:
                    G.add_edge(
                        sensor_id, 
                        sensor_id_2, 
                        distance=distance_m, 
                        dir=direction
                    )

In [8]:
manual_edges = [(11,23), (9,22), (9,23), (1,12), (2,14), (3,14),(4,14), (5,15), (8,20), (6,19), (6,16), (8,19), (7,20), (7,19), (6,5), (17,14), (15,14), (16,14)]

In [9]:
import geopy.distance

node_positions = nx.get_node_attributes(G, "pos")

for source, target in manual_edges:
    lon1, lat1 = node_positions[source]
    lon2, lat2 = node_positions[target]

    distance_m = geopy.distance.geodesic((lat1, lon1), (lat2, lon2)).m
    G.add_edge(source, target, distance=distance_m)

In [10]:
sensor_ids = list(G.nodes())
node_map = {sid: idx for idx, sid in enumerate(sensor_ids)}
num_nodes = len(sensor_ids)

sample_df = list(traffic_dic[sensor_ids[0]].values())[0]

exclude_cols = ['latitude', 'longitude', 'timestamp', 'time', 'miles', "avtime", "hour"]
base_features = [col for col in sample_df.columns if col not in exclude_cols]

time_features = ['hour_sin', 'hour_cos', 'day_sin', 'day_cos']
all_features = base_features + time_features

num_timestamps = len(sample_df)
num_features = len(all_features)

x_tensor = torch.zeros((num_nodes, num_features, num_timestamps), dtype=torch.float)

for sensor_id in sensor_ids:
    node_idx = node_map[sensor_id]
    traffic_df = list(traffic_dic[sensor_id].values())[0].copy()
    
    traffic_df['timestamp'] = pd.to_datetime(traffic_df['timestamp'])
    
    traffic_df['hour_sin'] = np.sin(2 * np.pi * traffic_df['timestamp'].dt.hour / 24.0)
    traffic_df['hour_cos'] = np.cos(2 * np.pi * traffic_df['timestamp'].dt.hour / 24.0)
    traffic_df['day_sin'] = np.sin(2 * np.pi * traffic_df['timestamp'].dt.dayofweek / 7.0)
    traffic_df['day_cos'] = np.cos(2 * np.pi * traffic_df['timestamp'].dt.dayofweek / 7.0)
    traffic_df["count_imputed"] = traffic_df["count_imputed"].apply(lambda x: 0 if x == False else 1)
    traffic_features_matrix = traffic_df[all_features].values.T
    x_tensor[node_idx, :, :] = torch.tensor(traffic_features_matrix, dtype=torch.float)

edge_list = []
edge_attr_list = []
for u, v, data in G.edges(data=True):
    edge_list.append([node_map[u], node_map[v]])
    edge_attr_list.append([data['distance']])

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_attr = torch.tensor(edge_attr_list, dtype=torch.float)

pyg_dataset = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr)

In [11]:
import plotly.graph_objects as go

num_nodes = len(node_map)
coords_traffic = np.zeros((num_nodes, 2))

for sensor_id, node_idx in node_map.items():
    lon, lat = G.nodes[sensor_id]['pos']
    coords_traffic[node_idx, 0] = lon
    coords_traffic[node_idx, 1] = lat

edge_t_to_t = pyg_dataset.edge_index


def visualize_traffic_spatial_graph(coords_traffic, edge_t_to_t, mapbox_style="carto-positron"):
    """
    Visualizes the traffic sensor spatial graph directly on a map using Plotly Scattermapbox.
    """
    fig = go.Figure()

    # --- DRAW TRAFFIC-TO-TRAFFIC EDGES ---
    t_edge_lon, t_edge_lat = [], []
    t_start_nodes = edge_t_to_t[0].numpy()
    t_end_nodes = edge_t_to_t[1].numpy()
    
    for src, dst in zip(t_start_nodes, t_end_nodes):
        # Plotly draws continuous paths; adding None breaks the line between distinct edges
        t_edge_lon.extend([coords_traffic[src, 0], coords_traffic[dst, 0], None])
        t_edge_lat.extend([coords_traffic[src, 1], coords_traffic[dst, 1], None])
        
    fig.add_trace(go.Scattermapbox(
        lon=t_edge_lon, lat=t_edge_lat,
        mode='lines',
        line=dict(width=1.5, color='rgba(50, 150, 250, 0.6)'),
        name='Traffic-to-Traffic Edges',
        hoverinfo='none'
    ))

    fig.add_trace(go.Scattermapbox(
        lon=coords_traffic[:, 0], lat=coords_traffic[:, 1],
        mode='markers',
        marker=dict(size=10, color='blue', opacity=0.85),
        name='Traffic Sensor Nodes',
        text=[f"Node Index: {i}<br>Sensor ID: {sensor_ids[i]}" for i in range(len(coords_traffic))],
        hoverinfo='text'
    ))

    center_lat = np.mean(coords_traffic[:, 1])
    center_lon = np.mean(coords_traffic[:, 0])

    fig.update_layout(
        title=dict(text='Spatio-Temporal Traffic Graph Topology', font=dict(size=18)),
        autosize=True,
        hovermode='closest',
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.7)"),
        mapbox=dict(
            style=mapbox_style,
            bearing=0,
            center=dict(lat=center_lat, lon=center_lon),
            pitch=0,
            zoom=12
        ),
        width=1100,
        height=750,
        margin=dict(r=0, t=40, l=0, b=0)
    )
    
    fig.show()

visualize_traffic_spatial_graph(coords_traffic, edge_t_to_t)

/tmp/ipykernel_2819732/3245259071.py:30: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/tmp/ipykernel_2819732/3245259071.py:38: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(


In [12]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# 1. GRAPH STRUCTURE + TEMPORAL FEATURE PREPARATION
# ==========================================
# Extract dimensions from the existing graph tensor
num_nodes_spatial = pyg_dataset.x.shape[0]
num_features_spatial = pyg_dataset.x.shape[1]
total_timestamps = pyg_dataset.x.shape[2]

# Build the spatial adjacency matrix from the PyG edge index
adj_matrix = np.zeros((num_nodes_spatial, num_nodes_spatial), dtype=np.float32)
edges = pyg_dataset.edge_index.numpy()
adj_matrix[edges[0], edges[1]] = 1.0
adj_matrix += np.eye(num_nodes_spatial, dtype=np.float32)

deg = np.sum(adj_matrix, axis=1)
deg_inv_sqrt = np.power(deg, -0.5, where=deg > 0)
deg_inv_sqrt[deg == 0] = 0.0
D_inv_sqrt = np.diag(deg_inv_sqrt)
normalized_adj = D_inv_sqrt @ adj_matrix @ D_inv_sqrt

# Convert traffic tensor to a time-major layout for windowing
traffic_np = pyg_dataset.x.numpy()
traffic_time_major = np.transpose(traffic_np, (2, 0, 1))  # (timestamps, nodes, features)

# Build a multivariate temporal dataframe for the PM2.5 branch
# This mirrors the logic from notebooks 4.5/4.6 while keeping it separate from the graph branch.
temporal_df = pd.merge(weather, air_qual, left_on="timestamp", right_on="time", how="inner")
temporal_df = temporal_df.sort_values("timestamp").set_index("timestamp")
temporal_df = temporal_df.asfreq("h")
temporal_df = temporal_df.drop(columns=["time"], errors="ignore")
idx_hour = temporal_df.index.hour
temporal_df["hour_sin"] = np.sin(2 * np.pi * idx_hour / 24.0)
temporal_df["hour_cos"] = np.cos(2 * np.pi * idx_hour / 24.0)
temporal_df = temporal_df.dropna()

temporal_feature_cols = [
    "pm2_5 (μg/m³)",
    "hour_sin",
    "hour_cos",
    "temperature_2m (°C)",
    "surface_pressure (hPa)",
    "wind_speed_100m (km/h)",
    "precipitation (mm)",
]
temporal_feature_cols = [col for col in temporal_feature_cols if col in temporal_df.columns]
target_col = "pm2_5 (μg/m³)"

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
scaled_temporal = scaler_X.fit_transform(temporal_df[temporal_feature_cols])
scaled_target = scaler_y.fit_transform(temporal_df[[target_col]])

# Optional extra branch for weather-only features; keep it disabled by default.
USE_WEATHER_BRANCH = False
if USE_WEATHER_BRANCH:
    weather_feature_cols = [
        "temperature_2m (°C)",
        "surface_pressure (hPa)",
        "wind_speed_100m (km/h)",
        "precipitation (mm)",
    ]
    weather_feature_cols = [col for col in weather_feature_cols if col in temporal_df.columns]
    scaler_X_weather = MinMaxScaler()
    scaled_weather = scaler_X_weather.fit_transform(temporal_df[weather_feature_cols])
else:
    scaled_weather = None

# Sliding-window generation for the two main branches
lookback = 24 * 3
forecast_steps = 24

traffic_timestamps = pd.to_datetime(temporal_df.index[:traffic_time_major.shape[0]])
traffic_series_map = pd.Series(range(len(traffic_timestamps)), index=traffic_timestamps)

x_traffic = []
x_temporal = []
x_weather_branch = []
y = []

for i in range(lookback, len(temporal_df) - forecast_steps + 1):
    current_time_window = temporal_df.index[i - lookback : i]
    try:
        traffic_indices = traffic_series_map.loc[current_time_window].values.astype(int)
        if len(traffic_indices) != lookback or np.isnan(traffic_indices).any():
            continue

        traffic_slice = traffic_time_major[traffic_indices, :, :]
        if traffic_slice.shape != (lookback, num_nodes_spatial, num_features_spatial):
            continue

        x_traffic.append(traffic_slice)
        x_temporal.append(scaled_temporal[i - lookback : i, :])
        if USE_WEATHER_BRANCH:
            x_weather_branch.append(scaled_weather[i - lookback : i, :])
        y.append(scaled_target[i : i + forecast_steps, 0])
    except KeyError:
        continue

x_traffic = np.array(x_traffic, dtype=np.float32)
x_temporal = np.array(x_temporal, dtype=np.float32)
y = np.array(y, dtype=np.float32)

if USE_WEATHER_BRANCH:
    x_weather_branch = np.array(x_weather_branch, dtype=np.float32)

print(f"Traffic branch shape: {x_traffic.shape}")
print(f"Temporal branch shape: {x_temporal.shape}")
print(f"Target shape: {y.shape}")
if USE_WEATHER_BRANCH:
    print(f"Weather branch shape: {x_weather_branch.shape}")

# Train/Test split
split = int(len(y) * 0.7)
x_train_traffic, x_test_traffic = x_traffic[:split], x_traffic[split:]
x_train_temporal, x_test_temporal = x_temporal[:split], x_temporal[split:]
y_train, y_test = y[:split], y[split:]

if USE_WEATHER_BRANCH:
    x_train_weather, x_test_weather = x_weather_branch[:split], x_weather_branch[split:]
else:
    x_train_weather = None
    x_test_weather = None

# Repeat the adjacency for each sample in the dataset
adj_train = np.repeat(np.expand_dims(normalized_adj, axis=0), len(x_train_traffic), axis=0)
adj_test = np.repeat(np.expand_dims(normalized_adj, axis=0), len(x_test_traffic), axis=0)

num_temporal_features = x_temporal.shape[-1]


2026-07-02 13:11:47.346707: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-02 13:11:47.388356: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-02 13:11:48.453970: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Traffic branch shape: (6879, 72, 24, 11)
Temporal branch shape: (6879, 72, 7)
Target shape: (6879, 24)


In [ ]:
# import tensorflow as tf
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout, Layer, Concatenate, Conv2D, MaxPooling2D, Flatten)

# # ==========================================
# # 2. DUAL-BRANCH SPATIO-TEMPORAL MODEL
# # ==========================================
# class GraphConvLayer(Layer):
#     def __init__(self, units, **kwargs):
#         super(GraphConvLayer, self).__init__(**kwargs)
#         self.units = units

#     def build(self, input_shape):
#         traffic_shape = input_shape[0] if isinstance(input_shape, list) else input_shape
#         self.w = self.add_weight(
#             shape=(traffic_shape[-1], self.units),
#             initializer="glorot_uniform",
#             trainable=True,
#             name="gcn_weight"
#         )
#         super(GraphConvLayer, self).build(input_shape)

#     def call(self, inputs, adj):
#         transformed = tf.matmul(inputs, self.w)
#         out = tf.einsum('bij,btjc->btic', adj, transformed)
#         return tf.nn.relu(out)


# # --- Branch 1: Graph-based traffic encoder ---
# traffic_inputs = Input(shape=(lookback, num_nodes_spatial, num_features_spatial), name="traffic_input")
# adj_inputs = Input(shape=(num_nodes_spatial, num_nodes_spatial), name="adjacency_input")

# t_conv1 = Conv2D(filters=32, kernel_size=(3, 1), padding='same', activation='relu')(traffic_inputs)
# t_conv1 = Dropout(0.3)(t_conv1)

# s_gcn = GraphConvLayer(units=32)(t_conv1, adj_inputs)
# s_gcn = Dropout(0.3)(s_gcn)

# pool_spatial = MaxPooling2D(pool_size=(2, 1))(s_gcn)
# flat_branch_spatial = Flatten()(pool_spatial)

# # --- Branch 2: Multivariate temporal PM2.5 encoder ---
# temporal_inputs = Input(shape=(lookback, num_temporal_features), name="temporal_input")

# temporal_lstm_1 = LSTM(64, return_sequences=True)(temporal_inputs)
# temporal_drop_1 = Dropout(0.3)(temporal_lstm_1)

# temporal_lstm_2 = LSTM(32, return_sequences=False)(temporal_drop_1)
# flat_branch_temporal = Dropout(0.3)(temporal_lstm_2)

# # --- Optional Branch 3: weather-only encoder ---
# if USE_WEATHER_BRANCH:
#     weather_inputs = Input(shape=(lookback, x_train_weather.shape[2]), name="weather_input")
#     weather_lstm = LSTM(32, return_sequences=False)(weather_inputs)
#     flat_branch_weather = Dropout(0.3)(weather_lstm)
#     merged_features = Concatenate()([flat_branch_spatial, flat_branch_temporal, flat_branch_weather])
# else:
#     merged_features = Concatenate()([flat_branch_spatial, flat_branch_temporal])

# # --- Fusion and output head ---
# dense_1 = Dense(64, activation="relu")(merged_features)
# drop_5 = Dropout(0.4)(dense_1)
# outputs = Dense(forecast_steps, name="forecast_output")(drop_5)

# # Build and compile
# model = Model(inputs=[traffic_inputs, adj_inputs, temporal_inputs] + ([weather_inputs] if USE_WEATHER_BRANCH else []), outputs=outputs)
# model.compile(optimizer="adam", loss="mse", metrics=["mae"])

# model.summary()


In [ ]:
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv

class TransformerLSTMModel(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_transformer_layers, hidden_dim, num_lstm_layers, output_dim, seq_len, dropout=0.1):
        super(TransformerLSTMModel, self).__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_encoder = nn.Parameter(torch.zeros(1, seq_len, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_transformer_layers)
        self.lstm = nn.LSTM(
            input_size=d_model, hidden_size=hidden_dim, num_layers=num_lstm_layers,
            batch_first=True, dropout=dropout if num_lstm_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        x = self.input_projection(src)
        x = x + self.pos_encoder[:, :src.size(1), :]
        x = self.dropout(x)
        transformer_out = self.transformer_encoder(x)
        lstm_out, (hn, cn) = self.lstm(transformer_out)
        out = lstm_out[:, -1, :]
        predictions = self.fc(self.dropout(out))
        return predictions


class TriBranchSpatioTemporalModel(nn.Module):
    def __init__(self, num_nodes, traffic_feat_dim, weather_feat_dim, seq_len, output_dim, 
                 transformer_kwargs, graph_hidden_dim=64, fusion_hidden_dim=128, dropout=0.2, use_weather=True):
        super().__init__()
        self.num_nodes = num_nodes
        self.output_dim = output_dim
        self.use_weather = use_weather
        
        # Branch 1: Temporal Transformer-LSTM Backbone (processes node data flatly)
        self.temporal_backbone = TransformerLSTMModel(
            input_dim=traffic_feat_dim, 
            output_dim=output_dim, 
            seq_len=seq_len, 
            **transformer_kwargs
        )
        
        # Branch 2: Topological Graph Convolution (GNN) Branch
        self.graph_conv1 = GCNConv(traffic_feat_dim, graph_hidden_dim)
        self.graph_conv2 = GCNConv(graph_hidden_dim, graph_hidden_dim)
        self.graph_fc = nn.Linear(graph_hidden_dim, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        
        # Branch 3: Weather/Global Condition Branch (Linear Context Encoder)
        if self.use_weather:
            self.weather_encoder = nn.Sequential(
                nn.Linear(weather_feat_dim * seq_len, 64),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(64, 32)
            )
            weather_out_dim = 32
        else:
            weather_out_dim = 0
        total_fused_dim = (num_nodes * output_dim) + (num_nodes * output_dim) + weather_out_dim
        
        self.fusion_head = nn.Sequential(
            nn.Linear(total_fused_dim, fusion_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden_dim, num_nodes * output_dim)
        )

    def forward(self, traffic_x, edge_index, weather_x=None):
        batch_size, N, S, F = traffic_x.shape
        
        # --- Branch 1: Transformer-LSTM Feature Extraction ---
        t_reshaped = traffic_x.view(batch_size * N, S, F)
        trans_out = self.temporal_backbone(t_reshaped)
        trans_flat = trans_out.view(batch_size, -1)
        
        # --- Branch 2: Spatial Graph Messaging Pass ---
        spatial_features = traffic_x.mean(dim=2)
        
        graph_outs = []
        for b in range(batch_size):
            g_x = spatial_features[b] # (N, F)
            g_hidden = self.relu(self.graph_conv1(g_x, edge_index))
            g_hidden = self.dropout(g_hidden)
            g_hidden = self.relu(self.graph_conv2(g_hidden, edge_index))
            g_out = self.graph_fc(g_hidden)
            graph_outs.append(g_out.view(-1))
            
        graph_flat = torch.stack(graph_outs, dim=0)
        
        # --- Branch 3: Weather Branch Context (Toggled Condition) ---
        if self.use_weather and weather_x is not None:
            w_flat = weather_x.reshape(batch_size, -1)
            weather_repr = self.weather_encoder(w_flat)
            fused_vector = torch.cat([trans_flat, graph_flat, weather_repr], dim=-1)
        else:
            fused_vector = torch.cat([trans_flat, graph_flat], dim=-1)
            
        predictions = self.fusion_head(fused_vector)
        return predictions.view(batch_size, N, self.output_dim)

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# --- DATA LOADING FOR TriBranchSpatioTemporalModel ---
x_train_traffic_model = torch.tensor(np.transpose(x_train_traffic, (0, 2, 1, 3)), dtype=torch.float32)
x_test_traffic_model = torch.tensor(np.transpose(x_test_traffic, (0, 2, 1, 3)), dtype=torch.float32)

x_train_weather = torch.tensor(x_train_temporal, dtype=torch.float32)
x_test_weather = torch.tensor(x_test_temporal, dtype=torch.float32)

y_train_nodes = torch.tensor(np.repeat(y_train[:, None, :], num_nodes_spatial, axis=1), dtype=torch.float32)
y_test_nodes = torch.tensor(np.repeat(y_test[:, None, :], num_nodes_spatial, axis=1), dtype=torch.float32)

train_dataset = TensorDataset(x_train_traffic_model, x_train_weather, y_train_nodes)
test_dataset = TensorDataset(x_test_traffic_model, x_test_weather, y_test_nodes)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, drop_last=False)

node_labels = [str(n) for n in sensor_ids]
edge_idx_tensor = edge_index

# --- MODEL INSTANTIATION ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TriBranchSpatioTemporalModel(
    num_nodes=num_nodes_spatial,
    traffic_feat_dim=num_features_spatial,
    weather_feat_dim=x_train_weather.shape[-1],
    seq_len=lookback,
    output_dim=forecast_steps,
    transformer_kwargs={
        "d_model": 64,
        "nhead": 4,
        "num_transformer_layers": 1,
        "hidden_dim": 64,
        "num_lstm_layers": 1,
        "dropout": 0.1,
    },
    graph_hidden_dim=64,
    fusion_hidden_dim=128,
    dropout=0.2,
    use_weather=True,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-5, verbose=True)

# --- TRAINING LOOP ---
epochs = 20
hist_train = []
hist_val = []

for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0

    for traffic_batch, weather_batch, y_batch in train_loader:
        traffic_batch = traffic_batch.to(DEVICE)
        weather_batch = weather_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        preds = model(traffic_batch, edge_idx_tensor.to(DEVICE), weather_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * traffic_batch.size(0)

    train_loss /= len(train_dataset)
    hist_train.append(train_loss)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for traffic_batch, weather_batch, y_batch in test_loader:
            traffic_batch = traffic_batch.to(DEVICE)
            weather_batch = weather_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            preds = model(traffic_batch, edge_idx_tensor.to(DEVICE), weather_batch)
            loss = criterion(preds, y_batch)
            val_loss += loss.item() * traffic_batch.size(0)

    val_loss /= len(test_dataset)
    hist_val.append(val_loss)
    scheduler.step(val_loss)

    print(f"Epoch {epoch:02d}/{epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

# Plot training history
plt.figure(figsize=(8, 4))
plt.plot(hist_train, label="Train Loss", color="royalblue")
plt.plot(hist_val, label="Val Loss", color="darkorange")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("TriBranch Spatio-Temporal Training")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_tri_branch_model(model, test_loader, edge_index, node_names, device):
    model.eval()
    all_preds, all_trues = [], []
    edge_index = edge_index.to(device)
    
    with torch.no_grad():
        for batch in test_loader:
            if len(batch) == 3:
                traffic_x, weather_x, y_true = batch
                weather_x = weather_x.to(device)
            else:
                traffic_x, y_true = batch
                weather_x = None
                
            traffic_x = traffic_x.to(device)
            output = model(traffic_x, edge_index, weather_x)
            
            all_preds.append(output.cpu().numpy())
            all_trues.append(y_true.numpy())
            
    all_preds = np.concatenate(all_preds, axis=0) # Shape: (Samples, Nodes, Horizon)
    all_trues = np.concatenate(all_trues, axis=0) # Shape: (Samples, Nodes, Horizon)
    
    metrics_summary = []
    for idx, name in enumerate(node_names):
        true_series = all_trues[:, idx, :].flatten()
        pred_series = all_preds[:, idx, :].flatten()
        
        mae = mean_absolute_error(true_series, pred_series)
        rmse = np.sqrt(mean_squared_error(true_series, pred_series))
        r2 = r2_score(true_series, pred_series)
        
        metrics_summary.append({
            "Sensor Node": name,
            "MAE": round(mae, 4),
            "RMSE": round(rmse, 4),
            "R² Score": round(r2, 4)
        })
        
    df_metrics = pd.DataFrame(metrics_summary)
    print("\n" + "="*20 + " TRI-BRANCH EVALUATION SUMMARY " + "="*20)
    print(df_metrics.to_string(index=False))
    return df_metrics, all_preds, all_trues

In [ ]:
evaluate_tri_branch_model(model, test_loader, edge_idx_tensor, node_labels, DEVICE)